# Agentic + RAG metrics: a working walkthrough

Every agentic and RAG metric in AuditKit, run live against real models and scored step by step.

| Part | What you see |
|---|---|
| 1 | A complete agent sample: tools, reference turns, target |
| 2 | **Single step** (`api:`): the model's raw tool-call request → turns → scores |
| 3 | How matching works: the exact bipartite pairing, call by call |
| 4 | **Multi-turn agent** (`agent:`): model + Python tools in a loop, every step logged, then scored |
| 5 | Every agentic metric on 9 recorded behaviours (what each metric catches) |
| 6 | **RAG pipeline** (`agent:`): retriever + generator, scored by retrieval, judge and lexical metrics |
| 7 | **All 14 metrics in one `evaluate()` call** |
| 8 | Fingerprint and cache: what makes a run "the same run" |

**Who does what.** The model only *writes* tool-call requests. AuditKit only *sends* samples and *scores*
what comes back. Executing tools and looping (part 4) is the agent's job, not AuditKit's.

Needs an OpenAI-compatible server. Defaults: Ollama at `localhost:11434` with `qwen3:8b` (agent, native
tool calls) and `gemma3:12b` (judge, so the agent is not judging itself). On vLLM/SGLang set `BASE` and the
model names; serve the agent model with a tool-call parser.

In [1]:
import json, re, threading, time, urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import pandas as pd
import auditkit as ak
from auditkit.trace import predicted_turns, to_turns, flatten, max_matching

BASE = "http://localhost:11434/v1"      # any OpenAI-compatible server
AGENT_MODEL = "qwen3:8b"                 # must support native tool calling
JUDGE_MODEL = "gemma3:12b"               # judge for task_completion and the RAG judge metrics
JUDGE = dict(judge_model=f"api:{JUDGE_MODEL}", max_tokens=4096,
             judge_model_args={"api_base": BASE, "api_key": "EMPTY", "timeout": 600})

with urllib.request.urlopen(BASE + "/models", timeout=5) as r:
    served = [m["id"] for m in json.load(r)["data"]]
assert AGENT_MODEL in served and JUDGE_MODEL in served, f"server has {served}"
print("AuditKit", ak.__version__, "| server models:", served)

pd.set_option("display.max_colwidth", 140, "display.width", 250)

def fmt_turns(turns):
    return " → ".join("[" + ", ".join(f"{c.name}({json.dumps(c.arguments, ensure_ascii=False)})" for c in t) + "]"
                      for t in turns) or "(no calls)"

def per_sample(result):
    return pd.DataFrame({p.sample_id: {s["name"]: s["value"] for s in p.metadata.get("scores", [])}
                         for p in result.predictions}).T.round(2)

AuditKit 1.1.0 | server models: ['gemma3:12b', 'qwen3:8b']


## 1. A complete agent sample

`expected_tool_calls` is a list of **turns**. A turn with two or more calls is a *parallel group*
(independent calls that belong in one response); a later turn depends on the earlier ones. `[]` means
"no tool should be called". A single `api:` call only shows the model's **next** turn, so single-step
samples hold only the first turn; the multi-turn agent in part 4 gets the full path.

In [2]:
def fn(name, desc, props, required):
    return {"type": "function", "function": {"name": name, "description": desc,
            "parameters": {"type": "object", "properties": props, "required": required}}}
S = {"type": "string"}
TOOLS = [
    fn("get_weather", "Current weather for a city.",
       {"city": S, "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, ["city"]),
    fn("get_time", "Current local time in a city.", {"city": S}, ["city"]),
    fn("convert_currency", "Convert an amount between ISO currency codes.",
       {"amount": {"type": "number"}, "from_currency": S, "to_currency": S}, ["amount", "from_currency", "to_currency"]),
    fn("search_flights", "Search flights by IATA codes and date (YYYY-MM-DD).",
       {"origin": S, "destination": S, "date": S}, ["origin", "destination", "date"]),
    fn("book_flight", "Book a flight by the id returned from search_flights.", {"flight_id": S}, ["flight_id"]),
]
c = lambda name, **args: {"name": name, "arguments": args}

sample = ak.Sample(
    id="warmer-then-time",
    input="Which is warmer right now, Paris or Rome? Then tell me the local time in the warmer city.",
    tools=TOOLS,
    expected_tool_calls=[[c("get_weather", city="Paris"), c("get_weather", city="Rome")],   # turn 1: parallel
                         [c("get_time", city="Rome")]],                                     # turn 2: dependent
    target="Rome is warmer; it's 18:05 there.",
)
print(json.dumps({k: v for k, v in sample.__dict__.items() if v not in (None, [], {}, "")}, indent=1, default=str)[:1500])

{
 "input": "Which is warmer right now, Paris or Rome? Then tell me the local time in the warmer city.",
 "target": "Rome is warmer; it's 18:05 there.",
 "id": "warmer-then-time",
 "kind": "generative",
 "tools": [
  {
   "type": "function",
   "function": {
    "name": "get_weather",
    "description": "Current weather for a city.",
    "parameters": {
     "type": "object",
     "properties": {
      "city": {
       "type": "string"
      },
      "unit": {
       "type": "string",
       "enum": [
        "celsius",
        "fahrenheit"
       ]
      }
     },
     "required": [
      "city"
     ]
    }
   }
  },
  {
   "type": "function",
   "function": {
    "name": "get_time",
    "description": "Current local time in a city.",
    "parameters": {
     "type": "object",
     "properties": {
      "city": {
       "type": "string"
      }
     },
     "required": [
      "city"
     ]
    }
   }
  },
  {
   "type": "function",
   "function": {
    "name": "convert_currency",
  

## 2. Single step through `api:`: request → model's tool calls → scores

`ToolCallAdapter()` sends `Sample.tools` as the API `tools` field. The server returns
`message.tool_calls`; all calls in one response become **one turn** (a parallel call if 2+).
Default (`exact`) matching is schema-aware: an optional argument the tool schema declares (e.g. a
model adding `unit="celsius"` unasked) does not break a match, but an undeclared one does.
`arg_mode="subset"` also allows undeclared extras; `_name` variants ignore arguments entirely.

In [3]:
single_step = [
    ak.Sample(id="single", input="What's the weather in Tokyo?", tools=TOOLS,
              expected_tool_calls=[[c("get_weather", city="Tokyo")]]),
    ak.Sample(id="parallel-2", input="Compare the current weather in Paris and Rome.", tools=TOOLS,
              expected_tool_calls=[[c("get_weather", city="Paris"), c("get_weather", city="Rome")]]),
    ak.Sample(id="parallel-mixed", input="What's the weather and the local time in London?", tools=TOOLS,
              expected_tool_calls=[[c("get_weather", city="London"), c("get_time", city="London")]]),
    ak.Sample(id="dependent", input="Find flights from DEL to BOM on 2026-10-01 and book the cheapest.", tools=TOOLS,
              expected_tool_calls=[[c("search_flights", origin="DEL", destination="BOM", date="2026-10-01")]]),
    ak.Sample(id="numeric", input="Convert 250 USD to EUR.", tools=TOOLS,
              expected_tool_calls=[[c("convert_currency", amount=250, from_currency="USD", to_currency="EUR")]]),
    ak.Sample(id="irrelevant", input="What is 17 multiplied by 3?", tools=TOOLS, expected_tool_calls=[]),
]
AGENT_SCORERS = [ak.ToolCallF1(), ak.ToolCallF1(arg_mode="subset"), ak.ToolCallF1(arg_mode="name"),
                 ak.TrajectoryMatch(arg_mode="subset"), ak.ParallelToolCalls(arg_mode="subset"),
                 ak.ToolCallValidity(), ak.RedundantToolCalls()]

t0 = time.time()
r_single = ak.evaluate(single_step, model=f"api:{AGENT_MODEL}", api_base=BASE, api_key="EMPTY",
                       adapter=ak.ToolCallAdapter(), scorers=AGENT_SCORERS,
                       config=ak.RunConfig(temperature=0.0, max_tokens=4096, timeout=900))
print(f"{len(single_step)} samples in {time.time() - t0:.0f}s, errors={r_single.errors}")

p = next(p for p in r_single.predictions if p.sample_id == "parallel-2")
print("\nRaw native tool_calls the server returned for 'parallel-2' (one response):")
print(json.dumps(p.context["trace"]["tool_calls"], indent=1)[:900])

for s, p in zip(single_step, r_single.predictions):
    print(f"\n[{s.id}] {s.input}\n  expected : {fmt_turns(to_turns(s.expected_tool_calls))}"
          f"\n  predicted: {fmt_turns(predicted_turns(p.raw_output or '', p.context))}"
          + (f"\n  answer   : {p.raw_output.strip()[:100]}" if (p.raw_output or '').strip() else ""))

/Users/shreeyansarora/miniforge3/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


6 samples in 67s, errors=[]

Raw native tool_calls the server returned for 'parallel-2' (one response):
[
 [
  {
   "id": "call_iey88fsc",
   "index": 0,
   "type": "function",
   "function": {
    "name": "get_weather",
    "arguments": "{\"city\":\"Paris\",\"unit\":\"celsius\"}"
   }
  },
  {
   "id": "call_ce7a86a3",
   "index": 1,
   "type": "function",
   "function": {
    "name": "get_weather",
    "arguments": "{\"city\":\"Rome\",\"unit\":\"celsius\"}"
   }
  }
 ]
]

[single] What's the weather in Tokyo?
  expected : [get_weather({"city": "Tokyo"})]
  predicted: [get_weather({"city": "Tokyo", "unit": "celsius"})]

[parallel-2] Compare the current weather in Paris and Rome.
  expected : [get_weather({"city": "Paris"}), get_weather({"city": "Rome"})]
  predicted: [get_weather({"city": "Paris", "unit": "celsius"}), get_weather({"city": "Rome", "unit": "celsius"})]

[parallel-mixed] What's the weather and the local time in London?
  expected : [get_weather({"city": "London"}), get_t

In [4]:
per_sample(r_single)[["tool_call_f1", "tool_call_f1_subset", "tool_call_f1_name", "trajectory_strict_subset",
                      "parallel_recall_subset", "parallel_precision_subset", "parallel_detection_subset",
                      "tool_call_validity", "redundant_tool_calls"]]

,tool_call_f1,tool_call_f1_subset,tool_call_f1_name,trajectory_strict_subset,parallel_recall_subset,parallel_precision_subset,parallel_detection_subset,tool_call_validity,redundant_tool_calls
single,1.0,1.0,1.0,1.0,NaN,NaN,1.0,1.0,0.0
parallel-2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
parallel-mixed,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
dependent,1.0,1.0,1.0,1.0,NaN,NaN,1.0,1.0,0.0
numeric,1.0,1.0,1.0,1.0,NaN,NaN,1.0,1.0,0.0
irrelevant,1.0,1.0,1.0,1.0,NaN,NaN,1.0,NaN,NaN


## 3. How matching works

Each predicted call is paired with at most one reference call by an **exact maximum bipartite matching**
(not greedy first-fit, which rejects correct answers when one call fits several references).
Precision = matched / predicted, recall = matched / expected. Below: the pairing for `parallel-mixed`
under exact and subset argument rules.

In [5]:
s = next(x for x in single_step if x.id == "parallel-mixed")
p = next(x for x in r_single.predictions if x.sample_id == "parallel-mixed")
refs, preds = flatten(to_turns(s.expected_tool_calls)), flatten(predicted_turns(p.raw_output or "", p.context))
for mode in ("exact", "subset"):
    m = max_matching(preds, refs, mode)
    print(f"arg_mode={mode}: matched {len(m)}/{len(refs)} expected, {len(m)}/{len(preds)} predicted")
    for i, r in enumerate(refs):
        got = f"{preds[m[i]].name}({preds[m[i]].arguments})" if i in m else "UNMATCHED"
        print(f"   ref {r.name}({r.arguments})  ->  {got}")

arg_mode=exact: matched 2/2 expected, 2/2 predicted
   ref get_weather({'city': 'London'})  ->  get_weather({'city': 'London'})
   ref get_time({'city': 'London'})  ->  get_time({'city': 'London'})
arg_mode=subset: matched 2/2 expected, 2/2 predicted
   ref get_weather({'city': 'London'})  ->  get_weather({'city': 'London'})
   ref get_time({'city': 'London'})  ->  get_time({'city': 'London'})


## 4. Multi-turn agent through `agent:`: the agent runs the loop, AuditKit scores the transcript

The agent below (plain Python, **outside** AuditKit) calls the model, executes the requested tools,
sends the results back, and repeats until the model answers. It sits behind an HTTP endpoint;
`model="agent:<url>"` makes AuditKit POST each sample there and score the returned transcript
(each assistant message with `tool_calls` = one turn). `task_completion` is judged by the
separate judge model.

In [6]:
WEATHER = {"paris": (14, "rain"), "rome": (23, "sunny"), "tokyo": (18, "cloudy")}
FLIGHTS = [{"flight_id": "AI-5", "price_usd": 120}, {"flight_id": "UK-102", "price_usd": 80}]
PY_TOOLS = {
    "get_weather": lambda city, unit="celsius": dict(zip(("temp_c", "conditions"), WEATHER.get(city.lower(), (20, "clear")))),
    "get_time": lambda city: {"local_time": "18:05"},
    "convert_currency": lambda amount, from_currency, to_currency: {"amount": round(amount * 0.92, 2)},
    "search_flights": lambda origin, destination, date: {"flights": FLIGHTS},
    "book_flight": lambda flight_id: {"status": "confirmed", "flight_id": flight_id},
}
STEP_LOG = []

def call_model(messages, tools):
    body = {"model": AGENT_MODEL, "messages": messages, "tools": tools, "temperature": 0, "max_tokens": 4096}
    req = urllib.request.Request(BASE + "/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=900) as r:
        return json.load(r)["choices"][0]["message"]

def run_agent(task, tools, max_steps=6):
    messages, log = [{"role": "user", "content": task}], [f"TASK: {task}"]
    for step in range(1, max_steps + 1):
        msg = call_model(messages, tools)
        calls = msg.get("tool_calls") or []
        if not calls:
            log.append(f"  step {step}: model answers -> {(msg.get('content') or '').strip()[:120]}")
            messages.append({"role": "assistant", "content": msg.get("content") or ""})
            break
        log.append(f"  step {step}: model requests {len(calls)} call(s) in one response" + (" (parallel)" if len(calls) > 1 else ""))
        messages.append({"role": "assistant", "content": msg.get("content") or "", "tool_calls": calls})
        for call in calls:
            name, args = call["function"]["name"], json.loads(call["function"]["arguments"] or "{}")
            result = PY_TOOLS[name](**args) if name in PY_TOOLS else {"error": "unknown tool"}
            log.append(f"      agent runs {name}({args}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": call.get("id"), "content": json.dumps(result)})
    STEP_LOG.append("\n".join(log))
    return {"output": messages[-1].get("content", ""), "messages": messages}

class AgentHTTP(BaseHTTPRequestHandler):
    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        data = json.dumps(run_agent(body["input"], body.get("tools") or [])).encode()
        self.send_response(200); self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data))); self.end_headers(); self.wfile.write(data)
    def log_message(self, *a): pass

agent_srv = ThreadingHTTPServer(("127.0.0.1", 0), AgentHTTP)
threading.Thread(target=agent_srv.serve_forever, daemon=True).start()
AGENT_URL = f"http://127.0.0.1:{agent_srv.server_address[1]}/run"
print("agent endpoint:", AGENT_URL)

agent endpoint: http://127.0.0.1:53392/run


In [7]:
multi_turn = [
    sample,   # the part-1 sample: parallel weather, then dependent get_time
    ak.Sample(id="search-then-book", tools=TOOLS, target="UK-102 booked",
              input="Find flights from DEL to BOM on 2026-10-01 and book the cheapest one.",
              expected_tool_calls=[[c("search_flights", origin="DEL", destination="BOM", date="2026-10-01")],
                                   [c("book_flight", flight_id="UK-102")]]),
    ak.Sample(id="no-tool-needed", tools=TOOLS, input="What is 12 + 30?", expected_tool_calls=[], target="42"),
]
STEP_LOG.clear(); t0 = time.time()
r_agent = ak.evaluate(multi_turn, model=f"agent:{AGENT_URL}", adapter=ak.ToolCallAdapter(),
                      scorers=AGENT_SCORERS + [ak.TaskCompletion(**JUDGE)], config=ak.RunConfig(timeout=1800))
print(f"{len(multi_turn)} agent runs in {time.time() - t0:.0f}s, errors={r_agent.errors}\n")
print("\n\n".join(STEP_LOG))
for s, p in zip(multi_turn, r_agent.predictions):
    print(f"\n[{s.id}]\n  expected : {fmt_turns(to_turns(s.expected_tool_calls))}"
          f"\n  predicted: {fmt_turns(predicted_turns(p.raw_output or '', p.context))}")

3 agent runs in 118s, errors=[]

TASK: Which is warmer right now, Paris or Rome? Then tell me the local time in the warmer city.
  step 1: model requests 2 call(s) in one response (parallel)
      agent runs get_weather({'city': 'Paris', 'unit': 'celsius'}) -> {'temp_c': 14, 'conditions': 'rain'}
      agent runs get_weather({'city': 'Rome', 'unit': 'celsius'}) -> {'temp_c': 23, 'conditions': 'sunny'}
  step 2: model requests 1 call(s) in one response
      agent runs get_time({'city': 'Rome'}) -> {'local_time': '18:05'}
  step 3: model answers -> Rome is currently warmer than Paris. Rome has a temperature of 23°C (sunny), while Paris is at 14°C (rainy). The local t

TASK: Find flights from DEL to BOM on 2026-10-01 and book the cheapest one.
  step 1: model requests 1 call(s) in one response
      agent runs search_flights({'destination': 'BOM', 'origin': 'DEL', 'date': '2026-10-01'}) -> {'flights': [{'flight_id': 'AI-5', 'price_usd': 120}, {'flight_id': 'UK-102', 'price_usd': 80}]}
  

In [8]:
per_sample(r_agent)[["tool_call_f1_subset", "tool_call_f1_name", "trajectory_strict_subset", "trajectory_in_order_subset",
                     "parallel_recall_subset", "parallel_precision_subset", "tool_call_validity",
                     "redundant_tool_calls", "task_completion"]]

,tool_call_f1_subset,tool_call_f1_name,trajectory_strict_subset,trajectory_in_order_subset,parallel_recall_subset,parallel_precision_subset,tool_call_validity,redundant_tool_calls,task_completion
warmer-then-time,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
search-then-book,1.0,1.0,1.0,1.0,NaN,NaN,1.0,0.0,1.0
no-tool-needed,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,1.0


## 5. What each agentic metric catches: 9 recorded behaviours of one task

Reference: `[weather(Paris), weather(Rome)] → [book(AZ-61)]`. Each row is a different recorded agent run,
scored offline (`model="precomputed"`) — no model calls, fully deterministic. Read down a column to see
which behaviours a metric can and cannot tell apart (e.g. `tool_call_f1` rates *serialized* and *wrongly
batched* as perfect; the trajectory and parallel metrics don't).

In [9]:
P, R, BOOK = c("get_weather", city="Paris"), c("get_weather", city="Rome"), c("book_flight", flight_id="AZ-61")
behaviours = {
    "correct":         [[P, R], [BOOK]],
    "serialized":      [[P], [R], [BOOK]],
    "wrongly_batched": [[P, R, BOOK]],
    "wrong_argument":  [[P, c("get_weather", city="Lyon")], [BOOK]],
    "missing_call":    [[P, R]],
    "looping":         [[P, R], [P, R], [BOOK]],
    "invalid_calls":   [[c("get_weather", city="Paris", unit="kelvin"), c("teleport", to="Rome")]],
    "irrelevant_ok":   [],
    "irrelevant_bad":  [[P]],
}
recorded = [ak.Sample(id=k, input="Which is warmer, Paris or Rome? Book AZ-61 if it's Rome.", tools=TOOLS,
                      expected_tool_calls=[] if k.startswith("irrelevant") else [[P, R], [BOOK]],
                      actual_output="done", actual_trace={"tool_calls": v}) for k, v in behaviours.items()]
r_rec = ak.evaluate(recorded, model="precomputed",
                    scorers=[ak.ToolCallF1(), ak.TrajectoryMatch(), ak.ParallelToolCalls(), ak.ToolCallValidity(),
                             ak.RedundantToolCalls()])
per_sample(r_rec)[["tool_call_precision", "tool_call_recall", "tool_call_f1", "trajectory_strict", "trajectory_in_order",
                   "parallel_recall", "parallel_precision", "parallel_detection", "tool_call_validity",
                   "redundant_tool_calls"]]

,tool_call_precision,tool_call_recall,tool_call_f1,trajectory_strict,trajectory_in_order,parallel_recall,parallel_precision,parallel_detection,tool_call_validity,redundant_tool_calls
correct,1.00,1.00,1.00,1.0,1.0,1.0,1.0,1.0,1.0,0.0
serialized,1.00,1.00,1.00,0.0,1.0,0.0,NaN,0.0,1.0,0.0
wrongly_batched,1.00,1.00,1.00,0.0,1.0,1.0,0.0,1.0,1.0,0.0
wrong_argument,0.67,0.67,0.67,0.0,0.0,0.0,NaN,1.0,1.0,0.0
missing_call,1.00,0.67,0.80,0.0,0.0,1.0,1.0,1.0,1.0,0.0
looping,0.60,1.00,0.75,0.0,1.0,1.0,1.0,1.0,1.0,0.4
invalid_calls,0.50,0.33,0.40,0.0,0.0,0.0,NaN,1.0,0.0,0.0
irrelevant_ok,1.00,1.00,1.00,1.0,1.0,NaN,NaN,1.0,NaN,NaN
irrelevant_bad,0.00,0.00,0.00,0.0,0.0,NaN,NaN,1.0,1.0,0.0


`NaN` = the metric had nothing to measure on that run (e.g. `parallel_precision` needs a turn with 2+
matched calls; validity/redundancy need at least one call). It is omitted, not scored 0, so it never drags
an average down.

## 6. RAG pipeline: retriever + generator behind `agent:`, scored by every RAG metric

A tiny knowledge base, a keyword retriever (top 3) and the model as generator, served as an endpoint that
returns `{"output", "contexts"}`. The returned contexts are what the metrics score — the retrieval the
pipeline *actually* did. `reference_contexts` holds the gold chunks for the deterministic ranking metrics;
`target` feeds `context_recall`, `context_coverage` and `answer_overlap`.

In [10]:
KB = [
    "Employee laptops are replaced every 3 years through the IT portal.",
    "Laptops must use full-disk encryption.",
    "The cafeteria is open from 8am to 3pm on weekdays.",
    "Guest wifi passwords rotate every Monday.",
    "Annual leave is 25 days per year, plus public holidays.",
    "Unused annual leave of up to 5 days can be carried over to the next year.",
]
STOP = set("the a an is are of to for and on in from per up can be what how how's does do i my our".split())

def retrieve(q, k=3):
    words = {w for w in re.findall(r"\w+", q.lower()) if w not in STOP}
    scored = sorted(KB, key=lambda d: -len(words & set(re.findall(r"\w+", d.lower()))))
    return scored[:k]

def generate(q, ctx):
    prompt = ("Answer the question using ONLY the context. If the context does not contain the answer, say you "
              "don't know.\n\nContext:\n" + "\n".join(f"- {x}" for x in ctx) + f"\n\nQuestion: {q}")
    body = {"model": AGENT_MODEL, "messages": [{"role": "user", "content": prompt}], "temperature": 0, "max_tokens": 4096}
    req = urllib.request.Request(BASE + "/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=900) as r:
        return json.load(r)["choices"][0]["message"]["content"]

class RagHTTP(BaseHTTPRequestHandler):
    def do_POST(self):
        q = json.loads(self.rfile.read(int(self.headers["Content-Length"])))["input"]
        ctx = retrieve(q)
        data = json.dumps({"output": generate(q, ctx), "contexts": ctx}).encode()
        self.send_response(200); self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data))); self.end_headers(); self.wfile.write(data)
    def log_message(self, *a): pass

rag_srv = ThreadingHTTPServer(("127.0.0.1", 0), RagHTTP)
threading.Thread(target=rag_srv.serve_forever, daemon=True).start()
RAG_URL = f"http://127.0.0.1:{rag_srv.server_address[1]}/run"

rag_samples = [
    ak.Sample(id="laptop-policy", input="How often are laptops replaced, and what security do they need?",
              reference_contexts=[KB[0], KB[1]], target="Laptops are replaced every 3 years. They must use full-disk encryption."),
    ak.Sample(id="leave-carryover", input="How many days of annual leave can I carry over?",
              reference_contexts=[KB[5]], target="Up to 5 days of unused annual leave can be carried over."),
    ak.Sample(id="not-in-kb", input="What is the parental leave policy?",
              reference_contexts=[], target="The knowledge base does not cover parental leave."),
]
RAG_SCORERS = [ak.RetrievalMetrics(k=3), ak.Faithfulness(**JUDGE), ak.ContextPrecision(**JUDGE),
               ak.ContextRecall(**JUDGE), ak.LexicalGroundedness(), ak.ContextCoverage(), ak.ContextOverlap(),
               ak.AnswerOverlap()]
t0 = time.time()
r_rag = ak.evaluate(rag_samples, model=f"agent:{RAG_URL}", scorers=RAG_SCORERS, config=ak.RunConfig(timeout=1800))
print(f"RAG run in {time.time() - t0:.0f}s, errors={r_rag.errors}")
for s, p in zip(rag_samples, r_rag.predictions):
    print(f"\n[{s.id}] {s.input}\n  retrieved: {p.context['trace'].get('retrieved_contexts')}\n  answer   : {(p.raw_output or '').strip()[:220]}")

RAG run in 70s, errors=[]

[laptop-policy] How often are laptops replaced, and what security do they need?
  retrieved: ['Employee laptops are replaced every 3 years through the IT portal.', 'Laptops must use full-disk encryption.', 'The cafeteria is open from 8am to 3pm on weekdays.']
  answer   : Laptops are replaced every 3 years through the IT portal, and they must use full-disk encryption.

[leave-carryover] How many days of annual leave can I carry over?
  retrieved: ['Unused annual leave of up to 5 days can be carried over to the next year.', 'Annual leave is 25 days per year, plus public holidays.', 'Employee laptops are replaced every 3 years through the IT portal.']
  answer   : The context states that "Unused annual leave of up to 5 days can be carried over to the next year." Therefore, the answer is **5 days**.

[not-in-kb] What is the parental leave policy?
  retrieved: ['Annual leave is 25 days per year, plus public holidays.', 'Unused annual leave of up to 5 days can be 

In [11]:
per_sample(r_rag)

,hit_rate@3,precision@3,recall@3,mrr@3,average_precision@3,ndcg@3,faithfulness,context_precision,context_relevance,context_recall,lexical_groundedness,context_coverage,context_overlap,answer_overlap
laptop-policy,1.0,0.67,1.0,1.0,1.0,1.0,1.00,1.0,0.67,1.0,0.88,0.92,1.0,1.00
leave-carryover,1.0,0.33,1.0,1.0,1.0,1.0,0.75,1.0,0.33,1.0,0.81,1.00,1.0,1.00
not-in-kb,NaN,NaN,NaN,NaN,NaN,NaN,1.00,0.0,0.00,0.0,0.25,0.25,1.0,0.62


Reading the table:
- `not-in-kb` has no gold chunks, so the ranking metrics skip it (no fake 0). A correct "the context doesn't
  say" answer is faithful (its one claim is supported) but `context_recall` is 0: the retriever brought back
  nothing about parental leave, which is exactly what recall should report.
- The **lexical** metrics are coarse: `context_overlap` counts a whole chunk as relevant if it shares any word
  with the answer (even "the"), so it can read 1.0 where `lexical_groundedness` reads low. Prefer the judge
  metrics for decisions; the lexical ones are cheap sanity signals.
- A judge's claim split is itself model-dependent: it may turn a phrase like "Therefore, the answer is 5 days"
  into a separate claim and mark it UNSUPPORTED (see the verdicts below). That is judge behaviour, visible
  because every claim and verdict is kept on the score.

The faithfulness **claims and verdicts** the judge produced are kept on each score:

In [12]:
for p in r_rag.predictions:
    for s in p.metadata.get("scores", []):
        if s["name"] == "faithfulness":
            print(f"[{p.sample_id}] faithfulness={s['value']:.2f}")
            for claim, verdict in zip(s["metadata"]["claims"], s["metadata"]["verdicts"]):
                print(f"   {verdict:12s} {claim}")

[laptop-policy] faithfulness=1.00
   SUPPORTED    Laptops are replaced every 3 years.
   SUPPORTED    Laptops are replaced through the IT portal.
   SUPPORTED    Laptops must use full-disk encryption.
[leave-carryover] faithfulness=0.75
   SUPPORTED    Unused annual leave can be carried over.
   SUPPORTED    The maximum amount of annual leave that can be carried over is 5 days.
   SUPPORTED    The carry-over applies to the next year.
   UNSUPPORTED  The answer is 5 days.
[not-in-kb] faithfulness=1.00
   SUPPORTED    The context does not contain information about the parental leave policy.


A deliberately **hallucinated** answer next to a faithful one, same question and contexts (recorded, judged live):

In [13]:
ctx = retrieve("How often are laptops replaced, and what security do they need?")
contrast = [
    ak.Sample(id="faithful", input="How often are laptops replaced?", actual_output="Employee laptops are replaced every 3 years through the IT portal.",
              target="Laptops are replaced every 3 years.", actual_trace={"retrieved_contexts": ctx}),
    ak.Sample(id="hallucinated", input="How often are laptops replaced?", actual_output="Laptops are replaced every year and employees also get a free phone.",
              target="Laptops are replaced every 3 years.", actual_trace={"retrieved_contexts": ctx}),
]
r_contrast = ak.evaluate(contrast, model="precomputed", scorers=[ak.Faithfulness(**JUDGE), ak.LexicalGroundedness(), ak.AnswerOverlap()])
per_sample(r_contrast)

,faithfulness,lexical_groundedness,answer_overlap
faithful,1.0,1.0,1.00
hallucinated,0.0,0.5,0.67


## 7. All 14 metrics in one `evaluate()` call

The runs above, re-scored together offline: every agentic and RAG metric in a single call over a mixed
dataset. Each metric scores the samples that have what it needs and skips the rest, so one run can hold
agent samples and RAG samples side by side.

In [14]:
def to_recorded(samples, result):
    out = []
    for s, p in zip(samples, result.predictions):
        out.append(ak.Sample(id=s.id, input=s.input, tools=s.tools, expected_tool_calls=s.expected_tool_calls,
                             reference_contexts=s.reference_contexts, target=s.target,
                             actual_output=p.raw_output or "", actual_trace=(p.context or {}).get("trace") or {}))
    return out

mixed = to_recorded(multi_turn, r_agent) + to_recorded(rag_samples, r_rag)
ALL_METRICS = [
    # agentic
    ak.ToolCallF1(arg_mode="subset"), ak.TrajectoryMatch(arg_mode="subset"), ak.ParallelToolCalls(arg_mode="subset"),
    ak.ToolCallValidity(), ak.RedundantToolCalls(), ak.TaskCompletion(**JUDGE),
    # RAG: ranking, judged, lexical
    ak.RetrievalMetrics(k=3), ak.Faithfulness(**JUDGE), ak.ContextPrecision(**JUDGE), ak.ContextRecall(**JUDGE),
    ak.LexicalGroundedness(), ak.ContextCoverage(), ak.ContextOverlap(), ak.AnswerOverlap(),
]
t0 = time.time()
r_all = ak.evaluate(mixed, model="precomputed", scorers=ALL_METRICS)
first_run_s = time.time() - t0
print(f"{len(mixed)} samples x {len(ALL_METRICS)} metrics in {first_run_s:.0f}s | errors={r_all.errors} | fingerprint={r_all.fingerprint}")
per_sample(r_all).T.fillna("–")   # metrics as rows; '–' = metric not applicable to that sample

6 samples x 14 metrics in 74s | errors=[] | fingerprint=6bdbfb1f64e20948


,warmer-then-time,search-then-book,no-tool-needed,laptop-policy,leave-carryover,not-in-kb
tool_call_precision_subset,1.0,1.0,1.0,–,–,–
tool_call_recall_subset,1.0,1.0,1.0,–,–,–
tool_call_f1_subset,1.0,1.0,1.0,–,–,–
tool_call_exact_subset,1.0,1.0,1.0,–,–,–
trajectory_strict_subset,1.0,1.0,1.0,–,–,–
trajectory_in_order_subset,1.0,1.0,1.0,–,–,–
parallel_recall_subset,1.0,–,–,–,–,–
parallel_precision_subset,1.0,–,–,–,–,–
parallel_detection_subset,1.0,1.0,1.0,–,–,–
tool_call_validity,1.0,1.0,–,–,–,–


In [15]:
pd.Series(r_all.headline).round(3).to_frame("headline (mean over samples that have the score)")

,headline (mean over samples that have the score)
tool_call_precision_subset,1.000
tool_call_recall_subset,1.000
tool_call_f1_subset,1.000
tool_call_exact_subset,1.000
trajectory_strict_subset,1.000
trajectory_in_order_subset,1.000
parallel_recall_subset,1.000
parallel_precision_subset,1.000
parallel_detection_subset,1.000
tool_call_validity,1.000


## 8. Fingerprint and cache

A run's fingerprint hashes everything that can change its results: the dataset (inputs, tools, reference
turns, gold contexts, recorded outputs and traces, **sample ids**), the model and its server, the adapter,
every metric's settings (including a judge's model **and server**), the run config, and the AuditKit
version. Credentials and timeouts are left out. An identical run is served from the disk cache.

In [16]:
t0 = time.time()
r_again = ak.evaluate(mixed, model="precomputed", scorers=ALL_METRICS)
print(f"identical run   : fingerprint={r_again.fingerprint}  same={r_again.fingerprint == r_all.fingerprint}  "
      f"took {time.time() - t0:.2f}s (first run {first_run_s:.0f}s -> cache hit)")

from auditkit.runspec import RunSpec, RunConfig
from auditkit.scenario import ListScenario
from auditkit.model import AutoModel
def fp(samples, metrics):
    return RunSpec(scenario=ListScenario(samples), model=AutoModel.resolve("precomputed"),
                   adapter=ak.ToolCallAdapter(), metrics=metrics, config=RunConfig()).fingerprint()

base = fp(mixed, ALL_METRICS)
renamed = [ak.Sample(**{**s.__dict__, "id": s.id + "-v2"}) for s in mixed]
other_server = [ak.Faithfulness(judge_model=f"api:{JUDGE_MODEL}", judge_model_args={"api_base": "http://other:8000/v1"})]
same_server = [ak.Faithfulness(judge_model=f"api:{JUDGE_MODEL}", max_tokens=4096,
                               judge_model_args={"api_base": BASE, "api_key": "different", "timeout": 5})]
rows = {
    "sample ids changed":             fp(renamed, ALL_METRICS) != base,
    "judge on another server":        fp(mixed, other_server) != fp(mixed, [ak.Faithfulness(**JUDGE)]),
    "tool_call_f1 exact vs subset":   fp(mixed, [ak.ToolCallF1()]) != fp(mixed, [ak.ToolCallF1(arg_mode="subset")]),
    "retrieval k=3 vs k=5":           fp(mixed, [ak.RetrievalMetrics(k=3)]) != fp(mixed, [ak.RetrievalMetrics(k=5)]),
    "judge api_key/timeout only (must NOT change)": fp(mixed, same_server) == fp(mixed, [ak.Faithfulness(**JUDGE)]),
    "metric order (must NOT change)": fp(mixed, ALL_METRICS[::-1]) == base,
}
pd.Series(rows).to_frame("behaves correctly")

identical run   : fingerprint=6bdbfb1f64e20948  same=True  took 0.00s (first run 74s -> cache hit)


,behaves correctly
sample ids changed,True
judge on another server,True
tool_call_f1 exact vs subset,True
retrieval k=3 vs k=5,True
judge api_key/timeout only (must NOT change),True
metric order (must NOT change),True


## Where each metric reads its evidence

| Metric | Needs on the sample | Reads from the run |
|---|---|---|
| `tool_call_f1`, `trajectory_match`, `parallel_tool_calls` | `expected_tool_calls` | trace `tool_calls` → `messages` → parsed text |
| `tool_call_validity` | `tools` | same calls |
| `redundant_tool_calls` | — | same calls |
| `task_completion` | judge; `target` optional | task + transcript (with tool results) + answer |
| `retrieval(k)` | `reference_contexts` | trace `retrieved_contexts` → `retrieval_context` |
| `faithfulness` | judge | answer (reasoning stripped) + retrieved contexts |
| `context_precision` / `context_relevance` | judge; `target` optional | retrieved contexts |
| `context_recall` | judge, `target` | retrieved contexts |
| `lexical_groundedness`, `context_overlap` | — | answer + retrieved contexts |
| `context_coverage` | `target` | retrieved contexts |
| `answer_overlap` | `target` | answer |

In [17]:
agent_srv.shutdown(); rag_srv.shutdown()